# Notebook 03: SMS and Communication Fraud Detection

Train a balanced TF-IDF baseline on the merged dataset. Hold out complete message texts so duplicates cannot leak between train and test.

In [1]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (classification_report, confusion_matrix,
                             precision_recall_curve, average_precision_score,
                             roc_auc_score)

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.sms_detector import build_sms_pipeline, save_model, analyse_message

communication = pd.read_csv(PROJECT_ROOT / "data" / "synthetic" / "communication_events.csv")
groups = communication["message"].str.casefold().str.strip()
splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(splitter.split(communication, communication["is_fraudulent"], groups=groups))
X_train = communication.iloc[train_idx]["message"]
X_test = communication.iloc[test_idx]["message"]
y_train = communication.iloc[train_idx]["is_fraudulent"]
y_test = communication.iloc[test_idx]["is_fraudulent"]
assert set(y_train.unique()) == {0, 1} and set(y_test.unique()) == {0, 1}
assert not set(groups.iloc[train_idx]) & set(groups.iloc[test_idx])
print("Rows:", len(communication), "unique messages:", groups.nunique())
print("Held-out messages:", groups.iloc[test_idx].nunique())
model = build_sms_pipeline()
model.fit(X_train, y_train)
probabilities = model.predict_proba(X_test)[:, 1]
predictions = (probabilities >= 0.5).astype(int)

print(classification_report(y_test, predictions, target_names=["legitimate", "fraud"]))
print("Confusion matrix:", confusion_matrix(y_test, predictions))
print("ROC-AUC:", round(roc_auc_score(y_test, probabilities), 4))
print("PR-AUC:", round(average_precision_score(y_test, probabilities), 4))

Rows: 1300 unique messages: 64
Held-out messages: 16
              precision    recall  f1-score   support

  legitimate       0.79      1.00      0.88        71
       fraud       1.00      0.92      0.96       251

    accuracy                           0.94       322
   macro avg       0.89      0.96      0.92       322
weighted avg       0.95      0.94      0.94       322

Confusion matrix: [[ 71   0]
 [ 19 232]]
ROC-AUC: 1.0
PR-AUC: 1.0


In [ ]:
precision, recall, thresholds = precision_recall_curve(y_test, probabilities)
plt.figure(figsize=(7, 4))
plt.plot(recall, precision)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("SMS fraud precision-recall curve")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

processed_dir = PROJECT_ROOT / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)
save_model(model, processed_dir / "sms_fraud_model.joblib")
print("Saved:", processed_dir / "sms_fraud_model.joblib")

In [3]:
examples = [
    "Your MoMo account will be suspended. Send your OTP immediately.",
    "Thank you for contacting support. Your OTP was used for a successful transaction.",
]
for i, message in enumerate(examples, 1):
    display(analyse_message(f"DEMO{i:03d}", message, model))

{'event_id': 'DEMO001',
 'fraud_probability': 0.8672,
 'detected_behaviours': ['urgency', 'threats', 'credential_request'],
 'classification': 'SUSPICIOUS'}

{'event_id': 'DEMO002',
 'fraud_probability': 0.0299,
 'detected_behaviours': [],
 'classification': 'LEGITIMATE'}